# **Day 2 — Text Representation: TF-IDF & Embeddings**

*After learning how to clean and structure our data in day 1, we move on to transforming our text into numeric vectors with TF-IDF and Embeddings.*

## **TF-IDF**

TF-IDF is a frequency based statistical method used in natural language preprocessing and information retrieval to **evaluate the impotance of words in a document in relation to a larger collection of documents**. It uses **Bag-of-words** to represent a document by which words it contains and how often, ignoring order. **Term Frequency** (*TF*) measures how frequent a word is in a document, **Inverse Document Frequency** (*IDF*) increases the weight of  rare words across documents while decreasing the weigth of common words.


***Term Frequency - TF***

$$\text{TF}(\text{word}, \text{doc}) = \frac{\text{count of word in doc}}{\text{total words in doc}}$$


***Inverse Document Frequency - IDF***

$$\text{IDF}(\text{word}, \text{corpus}) = \log\left(\frac{\text{total number of docs}}{\text{number of docs containing word}}\right)$$

***TF-IDF***

$$\text{TF-IDF}(\text{word}, \text{doc}, \text{corpus}) = \text{TF}(\text{word}, \text{doc}) \times \text{IDF}(\text{word}, \text{corpus})$$

## **Word Embeddings**

Word embeddings are static,meaning based numerical representations of words that enable ML modes to process and understand natural language. Early embedding techniques such as *Word2Vec* and *GloVe* **represent each word using a single fixed vector regardless of the context in which it appears**. To adress this limitation, **contextual embeddings** where introduced. With Contextual embeddings, the **representation of a word changes dynamically depending on its surrounding words or linguistic context**.

## **Static vs. Dynamic embeddings**

Static embeddings give evrey word a fixed numerical vector, while contextual embeddings change a word's vector based on its surrounding sentence. Static embeddings require more computational power, run slower and handle Polysemy - multiple meanings - efficiently as opposed to dynamic embeddings.

## **Imports**

In [1]:
import pandas as pd
from datasets import load_dataset
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## **Uploading the  AG News Dataset**

In [2]:
from google.colab import drive

drive.mount('/content/drive')

train_path="/content/drive/MyDrive/train.csv"


df = pd.read_csv(train_path, header=0, names=["label", "title", "description"])
df["text"] = df["title"] + " " + df["description"]


print("Extraction complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extraction complete!


## **Full Cleaning Pipeline**

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
  # lowercase
  text= text.lower()
  # remove punctuation
  text= text.translate(str.maketrans('', '', string.punctuation))
  # tokenize
  tokens = word_tokenize(text)
  # remove stopwords
  tokens = [word for word in tokens if word not in stop_words]
  # lemmatize
  tokens= [lemmatizer.lemmatize(word) for word in tokens]

  return tokens

df["cleaned_tokens"] = df["text"].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [4]:
df["cleaned_text"] = df["cleaned_tokens"].apply(lambda tokens: ' '.join(tokens))

X_train, X_val, y_train, y_val = train_test_split(df["cleaned_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

pred = clf.predict(X_val_tfidf)

tfidf_accuracy = accuracy_score(y_val, pred)
tfidf_f1 = f1_score(y_val, pred, average='macro')

print("TF-IDF Logistic Regression - Accuracy:", tfidf_accuracy)
print("F1-macro Score:", tfidf_f1)
print(classification_report(y_val, pred))


TF-IDF Logistic Regression - Accuracy: 0.9110416666666666
F1-macro Score: 0.9108503481840822
              precision    recall  f1-score   support

           1       0.92      0.90      0.91      6000
           2       0.95      0.97      0.96      6000
           3       0.88      0.89      0.88      6000
           4       0.89      0.89      0.89      6000

    accuracy                           0.91     24000
   macro avg       0.91      0.91      0.91     24000
weighted avg       0.91      0.91      0.91     24000



### **Categories**

1 - World

2 - Sports

3 - Business

4 - Sci/Tech


### **Observations**

- **Sports class** is the easiest category for the model to flag, scoring the highest precision, recall and F1 scores across all 4 classes. This is possibly due to sports articles containing highly distinctive vocabulary(team names, player names, scores) that TF-IDF is well suited to pick up on.
- **Business class** is the hardest category for the model to detect which could overlap with World category vacobulary.
- **Macro avd & weighted avg** are identical because of the balanced nature of the dataset.

## **Pre-trained Word Embeddings - Nearest Neighbours**

In [5]:
!pip install gensim

In [6]:
import gensim.downloader as api

word_vectors = api.load("glove-wiki-gigaword-100")
sample_words = ["stock", "team", "election", "computer"]

for word in sample_words:
  print(f"\nNearest neighbours of '{word}':")
  similar = word_vectors.most_similar(word, topn=5)
  for w, s in similar:
    print(f"{w} : {s:.3f}")


Nearest neighbours of 'stock':
shares : 0.853
stocks : 0.831
market : 0.799
exchange : 0.785
trading : 0.763

Nearest neighbours of 'team':
teams : 0.852
squad : 0.785
football : 0.772
players : 0.766
coach : 0.765

Nearest neighbours of 'election':
elections : 0.942
vote : 0.847
polls : 0.829
electoral : 0.828
presidential : 0.815

Nearest neighbours of 'computer':
computers : 0.875
software : 0.837
technology : 0.764
pc : 0.737
hardware : 0.729


The above reuslts show the most similar, semantically related 5 words to each word from the sample list extracted from the AG News dataset, and how sure the model is about the result. All of the results show the strong ability of the model to categorize and group words based on meaning.

## **TF-IDF vs. LSTM/Transformer Comparison**

In [7]:
comparison = pd.DataFrame({
    "Model": ["TF-IDF + Logistic Regression", "LSTM (Day 3 Baseline)", "DistilBERT Fine-tuned Transformer"],
    "Accuracy": [tfidf_accuracy, 0.9182, 0.9455],
    "F1-macro": [tfidf_f1, None, 0.9455]
})

print(comparison)

                               Model  Accuracy  F1-macro
0       TF-IDF + Logistic Regression  0.911042   0.91085
1              LSTM (Day 3 Baseline)  0.918200       NaN
2  DistilBERT Fine-tuned Transformer  0.945500   0.94550


### **Why TF-IDF performs reasonably well here?**

AG News is a topic classification task where distinctive vocabulary  strongly signals the correct category. **TF-IDF + Logistic Regression** provide a *strong fast interpretable baseline* compared to the LSTM and given that it does not require GPU or epochs training. The gap to DistilBERT is attributed to DitilBERT's contextual embeddings, which capture word meaning and order, that the TF-IDF's frequency based presentation cannot do. Additionally, TF-IDF treats every word independentlyand captures no semantic relationships.

### **Chosen Representation for this project**

**DistilBERT** was selected as the production model for this project, since it achieved the highest accuracy and F1-macro of all three approaches tested.